# 第五阶段：智能指针

## 实验 9：Ownership 选择与 Native SDK 收尾

本实验整合第五阶段的 owner/borrower 语义，并把它落实到 Native SDK 内部对象图与 C ABI handle。目标不是“所有资源都换成智能指针”，而是为每条对象关系选择最简单、最准确的类型。

本实验依次完成：

1. 使用 AddressSanitizer/UBSan 对比 raw pointer double-free 与 RAII；
2. 从函数签名翻译 borrow、transfer、shared ownership 和 observation；
3. 比较直接成员、`unique_ptr` 与 `shared_ptr` 的选择条件；
4. 构建内部自动析构、外部只有 `create/destroy` 的 SDK runtime；
5. 完成第五阶段验收，并把问题自然交给下一阶段的 move semantics。

运行方式：重启 `xcpp23` kernel 后从上到下执行安全代码。会触发 double-free 的独立程序只在终端中运行，不要粘贴到 Notebook 执行。

In [ ]:
// 本步骤：引入智能指针、定宽状态、字符串、断言和移动所需的标准库。
#include <cassert>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <memory>
#include <string>
#include <type_traits>
#include <utility>

### 1. Sanitizer 对比：runtime bug 与 compile-time restriction

仓库提供了两个独立程序：

- [raw pointer double-free](./src/09_raw_double_free.cpp)：故意对同一地址执行两次 `delete`；
- [`unique_ptr` 安全版本](./src/09_unique_ptr_safe.cpp)：由 RAII owner 自动执行唯一一次清理。

从第五阶段目录运行：

```bash
clang++ -std=c++23 -Wall -Wextra -Wpedantic -g -O0 \
    -fsanitize=address,undefined \
    src/09_raw_double_free.cpp \
    -o /tmp/09_raw_double_free
/tmp/09_raw_double_free
```

预期 AddressSanitizer 报告 `attempting double-free` 并终止程序。这是 raw ownership 约定被破坏后的 runtime bug。

再运行 RAII 版本：

```bash
clang++ -std=c++23 -Wall -Wextra -Wpedantic -g -O0 \
    -fsanitize=address,undefined \
    src/09_unique_ptr_safe.cpp \
    -o /tmp/09_unique_ptr_safe
/tmp/09_unique_ptr_safe
```

预期只出现一次构造和一次析构，Sanitizer 不报告错误。更重要的是，下面的 illegal state 无法编译：

```cpp
auto first = std::make_unique<User>();
auto second = first; // error：unique_ptr 的复制操作已删除
```

RAII type 不会消灭所有 bug，但能把“两个 owner 都认为自己应该 delete”的状态从运行期风险提前变成编译期限制。

### 2. 从签名翻译 ownership contract

先准备一个可观察的 `User`。后续代码会实际调用每类接口，确认类型表达的 ownership 行为。

In [ ]:
// 本步骤：定义带可变状态和生命周期计数的 User。
class User
{
public:
    User(int id, std::string name)
        : id_(id), name_(std::move(name))
    {
        // 构造开始 User 生命周期，供 ownership 接口验证。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 最后一个 owner 结束时析构，borrower 不参与释放。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    int id() const noexcept
    {
        // 返回普通值，不暴露 ownership。
        return id_;
    }

    const std::string &name() const noexcept
    {
        // 返回只读字符串借用，有效期不超过 User。
        return name_;
    }

    void rename(std::string name)
    {
        // 修改业务状态，不改变调用方的 ownership。
        name_ = std::move(name);
    }

    static int live_count() noexcept
    {
        // 返回当前存活对象数，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    std::string name_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：定义只读借用接口；User 必须存在，函数不参与 ownership。
void read_user(const User &user)
{
    // 只在调用期间读取对象。
    std::cout << "read " << user.id() << ": " << user.name() << '\n';
}

// 本步骤：定义可写借用接口；User 必须存在，owner 保持不变。
void rename_user(User &user, std::string name)
{
    // 通过 mutable reference 更新现有对象。
    user.rename(std::move(name));
}

// 本步骤：定义可空、非拥有 pointer 接口；具体生命周期由 contract 约束。
void maybe_read_user(const User *user)
{
    // nullable pointer 必须先检查再解引用。
    if (user != nullptr)
    {
        read_user(*user);
    }
}

// 本步骤：定义返回唯一 ownership 的 factory。
std::unique_ptr<User> create_user(int id, std::string name)
{
    // 返回值把新 User ownership 转移给调用方。
    return std::make_unique<User>(id, std::move(name));
}

// 本步骤：定义消费唯一 ownership 的 sink。
void consume_user(std::unique_ptr<User> user)
{
    // 按值参数取得 ownership，并在函数返回时释放 User。
    assert(user);
    std::cout << "consume " << user->name() << '\n';
}

// 本步骤：定义参与 shared ownership 的函数。
void share_user(std::shared_ptr<User> user)
{
    // 参数是本次调用期间的另一个 strong owner。
    assert(user);
    std::cout << "shared owners inside f = " << user.use_count() << '\n';
}

// 本步骤：定义不参与 ownership 的 shared-object observer。
void observe_user(std::weak_ptr<User> observer)
{
    // lock 成功时只在当前分支临时延长生命周期。
    if (std::shared_ptr<User> user = observer.lock())
    {
        std::cout << "observe " << user->name() << '\n';
    }
}

In [ ]:
// 本步骤：依次调用不同签名，从状态变化验证 ownership contract。
{
    const int before = User::live_count();
    auto unique_owner = create_user(1, "Alice");

    // reference 与 pointer 只借用，unique_owner 仍保持非空。
    read_user(*unique_owner);
    rename_user(*unique_owner, "Alicia");
    maybe_read_user(unique_owner.get());
    maybe_read_user(nullptr);
    assert(unique_owner);

    // std::move 明确把唯一 ownership 交给 consume_user。
    consume_user(std::move(unique_owner));
    assert(!unique_owner);
    assert(User::live_count() == before);

    // shared_ptr 副本使函数参与 shared ownership；weak_ptr 只观察。
    auto shared_owner = std::make_shared<User>(2, "Bob");
    std::weak_ptr<User> observer = shared_owner;
    share_user(shared_owner);
    assert(shared_owner.use_count() == 1);
    observe_user(observer);

    // 最后一个 strong owner 消失后，weak observer 不会阻止析构。
    shared_owner.reset();
    assert(observer.expired());
    assert(User::live_count() == before);
}

应能自然翻译这些类型：

| 签名 | 语义 |
| --- | --- |
| `void f(const User &user)` | read-only borrow，必须存在 |
| `void f(User &user)` | mutable borrow，必须存在 |
| `void f(User *user)` | nullable/non-owning pointer，ownership 仍需 contract 说明 |
| `void f(unique_ptr<User> user)` | ownership transfer into `f` |
| `unique_ptr<User> create()` | ownership transfer to caller |
| `void f(shared_ptr<User> user)` | `f` participates in shared ownership |
| `weak_ptr<User>` | non-owning observation of a shared object |

类型表达的是 contract，而不只是调用语法。

### 3. 所有对象都用 `shared_ptr` 是 ownership smell

如果对象图实际是：

```text
Runtime
 ├── owns Engine
 ├── owns Model
 └── owns Buffer
```

那么三个 `shared_ptr` 会无故引入 control block、引用计数和模糊的共享生命周期。第一选择应先考虑直接成员；只有需要 optional lifetime、动态替换、多态或 incomplete-type 隔离时，才考虑 `unique_ptr`。只有多个实体确实需要独立延长生命周期时，才使用 `shared_ptr`。

In [ ]:
// 本步骤：定义始终与 Runtime 同生共死的直接成员类型。
class Engine
{
public:
    std::int32_t execute() const noexcept
    {
        // Engine 只执行计算，不关心自己由哪种 owner 持有。
        return 42;
    }
};

class Model
{
public:
    std::int32_t version() const noexcept
    {
        // Model 以值成员存在，不需要独立堆生命周期。
        return 1;
    }
};

class Buffer
{
public:
    std::size_t size() const noexcept
    {
        // Buffer 生命周期完全属于 Runtime。
        return 256;
    }
};

// 本步骤：让 Runtime 直接拥有三个始终存在、无需多态的子对象。
class Runtime
{
public:
    std::int32_t execute() const
    {
        // 普通成员调用只借用内部对象，不需要引用计数。
        assert(model_.version() == 1);
        assert(buffer_.size() == 256);
        return engine_.execute();
    }

private:
    Engine engine_;
    Model model_;
    Buffer buffer_;
};

In [ ]:
// 本步骤：运行直接成员模型，确认无需动态分配或显式清理。
{
    Runtime runtime;

    // Runtime 构造时成员自动构造，业务方法通过引用隐式借用它们。
    assert(runtime.execute() == 42);

    // 离开作用域时成员按逆序自动析构，不需要 unique_ptr 或 destroy 成员函数。
}

选择优先级不是绝对性能排名，而是复杂度提示：

```text
直接成员
   │ 最简单：生命周期完全属于父对象
   ▼
unique_ptr
   │ 动态、可选、可替换、多态或实现隔离
   ▼
shared_ptr
     多个实体必须独立延长生命周期，需说明理由
```

不要从“raw pointer 可能危险”推导出“所有东西都必须 smart pointer”。更准确的规则是：选择最简单、最准确表达 ownership 的类型。

### 4. Native SDK：外部一个 handle，内部 ownership tree 自动清理

C ABI 只看到 opaque handle 与成对 lifecycle 函数，内部 `sdk_runtime` 直接拥有 C++ `Runtime`。销毁外层 handle 会递归触发所有成员的 RAII 清理。

In [ ]:
// 本步骤：定义只在 C++ 实现中可见的 opaque handle 完整布局。
struct sdk_runtime
{
    // Runtime 作为直接成员拥有其完整内部对象图。
    Runtime impl;
};

// 本步骤：实现 C ABI factory，把 owning handle 返回给调用方。
extern "C" sdk_runtime *sdk_runtime_create() noexcept
{
    try
    {
        // new 被限制在 ABI lifecycle boundary，并与 destroy 成对。
        return new sdk_runtime{};
    }
    catch (...)
    {
        // C++ 异常不能跨 C ABI，创建失败转换为 nullptr。
        return nullptr;
    }
}

// 本步骤：实现同步借用 handle 的 C ABI 业务函数。
extern "C" std::int32_t sdk_runtime_execute(const sdk_runtime *runtime) noexcept
{
    // C pointer 可空，边界验证失败时返回稳定状态值。
    if (runtime == nullptr)
    {
        return -1;
    }

    // 函数只借用 impl；ownership 仍属于外部 handle contract。
    return runtime->impl.execute();
}

// 本步骤：实现消费 handle ownership 的 C ABI 销毁函数。
extern "C" void sdk_runtime_destroy(sdk_runtime *runtime) noexcept
{
    // delete 外层 handle 会自动析构 Runtime 及其整个 ownership tree。
    delete runtime;
}

In [ ]:
// 本步骤：模拟 C 调用方验证 create、borrow execute 和 destroy。
{
    // create 返回 owning raw handle；调用方必须最终执行一次 destroy。
    sdk_runtime *runtime = sdk_runtime_create();
    assert(runtime != nullptr);

    // execute 只同步借用 handle，调用后 ownership 不变。
    assert(sdk_runtime_execute(runtime) == 42);
    assert(sdk_runtime_execute(nullptr) == -1);

    // destroy 消费 ownership；原地址随后失效，不能重复使用。
    sdk_runtime_destroy(runtime);
    runtime = nullptr;
}

绝不能直接暴露：

```cpp
extern "C" std::unique_ptr<Runtime> sdk_runtime_create();
```

`unique_ptr`、`shared_ptr` 和 `weak_ptr` 涉及 C++ 模板实例化、对象布局、构造析构、control block、标准库实现及编译器/runtime compatibility。C ABI 应保持为 pointer/handle、定宽状态码和显式 `create/destroy`。

```text
Internal C++
direct member / unique_ptr / shared_ptr / weak_ptr / RAII
                         │
══════════════ C ABI Boundary ══════════════
                         │
opaque pointer / handle / status / create / destroy
```

### 5. 第五阶段 ownership 思维模型

```text
                 Object Relationship
                        │
              ┌─────────┴─────────┐
              │                   │
          Ownership             Borrow
              │                   │
       ┌──────┴──────┐       ┌────┴────┐
       │             │       │         │
 unique_ptr      shared_ptr  reference pointer
                         │
                      weak_ptr
                      observer
```

更准确地说，`weak_ptr` 不属于 ownership，而是专门观察 shared object 的 non-owning edge。每次设计从五个问题开始：

1. Who owns it?
2. Can ownership transfer?
3. Can multiple owners extend lifetime?
4. Can it be absent?
5. Who borrows it, and for how long?

## 实验结论与阶段验收

完成本实验后，应能不查资料回答：

1. `unique_ptr` 为什么是 Native SDK 内部默认的动态 ownership 类型？—— single owner、deterministic destruction、低成本、支持显式转移。
2. 为什么 `auto second = first;` 不能复制 `unique_ptr`？—— unique ownership 不允许出现两个 owner。
3. `auto second = std::move(first);` 发生了什么？—— ownership 从 `first` 转移给 `second`，`first` 变空。
4. `shared_ptr` 为什么成本更高？—— 它需要 control block、reference counting 和共享生命周期协调。
5. cyclic `shared_ptr` 为什么泄漏？—— 环内 strong count 永远无法归零。
6. `weak_ptr` 有什么意义？—— 观察 shared object，但不参与 ownership，使用前通过 `lock()` 检查。
7. 为什么 `process(const Engine&)` 通常优于 `process(const unique_ptr<Engine>&)`？—— 函数只借用 Engine，不应依赖调用方的 ownership container。
8. 如何为 Runtime 的 Engine、Model、Buffer 选择类型？—— 从 owner、转移、共享、可空和借用时间出发，依次考虑直接成员、`unique_ptr`、`shared_ptr` 和 non-owning reference/pointer。

第五阶段最终原则：**选择最简单、最准确表达 ownership 和 lifetime contract 的类型。**

下一阶段将回答这里刻意留下的问题：`unique_ptr` 不能 copy，为什么 ownership 却能够通过 `std::move` 转移？